# When Do Neural Networks Stop Seeing Objects and Start Representing Their Affordances?

**MCNB Statistics Project — Presentation Notebook**

This notebook implements a project-oriented **Representational Similarity Analysis (RSA)** pipeline for comparing artificial neural network representations with human fMRI ROI responses.

## Core idea
Humans perceive objects not only by their visual appearance, but also by the actions they afford. We test whether progressively deeper ANN layers become more similar to higher-level cortical representations involved in object understanding and object-action processing.

## Planned comparisons

| Component | Choice |
|---|---|
| Models | ResNet-50, DINOv2 ViT-B, OpenCLIP ViT-B/32 |
| Layers | early, middle, late |
| Brain ROIs | visual object-processing ROI, parietal / affordance-related ROI |
| Method | Representational Similarity Analysis (RSA) |
| Main output | Layer-wise RSA plot comparing model × layer × ROI |

> **Important:** This notebook assumes that fMRI data are already preprocessed into beta / voxel-response matrices. It does **not** preprocess raw fMRI data.

## 1. Research question

**When do neural networks stop merely representing what objects look like and start representing what objects are for?**

We operationalize this question using RSA:

1. Extract layer-wise ANN features for the same stimulus images shown to human participants.
2. Convert ANN features into model RDMs.
3. Convert fMRI beta responses within each ROI into brain RDMs.
4. Compare model RDMs and brain RDMs with Spearman correlation.

### Hypothesis

- Early layers should align more strongly with visual object-processing ROIs.
- Later layers should align more strongly with higher-level/parietal object-action ROIs.
- OpenCLIP may show stronger late-layer alignment with semantic or affordance-related ROIs than DINOv2 or ResNet-50 because it was trained with image-text supervision.

In [ ]:
# ============================================================
# 2. Install dependencies
# ============================================================
# In Colab, uncomment these lines if packages are missing.

# !pip install -q torch torchvision timm open_clip_torch scikit-learn scipy pandas matplotlib seaborn pillow tqdm

In [ ]:
# ============================================================
# 3. Imports and global settings
# ============================================================

import os
import json
import math
import random
from pathlib import Path
from collections import OrderedDict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torchvision.models as tv_models
from torchvision import transforms

from scipy.spatial.distance import pdist, squareform
from scipy.stats import spearmanr, pearsonr
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

PROJECT_DIR = Path(".")
DATA_DIR = PROJECT_DIR / "data"
RESULTS_DIR = PROJECT_DIR / "results"
FIGURES_DIR = PROJECT_DIR / "figures"

for d in [DATA_DIR, RESULTS_DIR, FIGURES_DIR]:
    d.mkdir(exist_ok=True)

## 4. Expected data structure

For this presentation notebook to run, prepare the following files.

```text
project/
├── Project.ipynb
├── data/
│   ├── stimulus_table.csv
│   ├── images/
│   │   ├── image_001.jpg
│   │   ├── image_002.jpg
│   │   └── ...
│   ├── brain_visual.npy
│   └── brain_parietal.npy
├── results/
└── figures/
```

### Required files

#### `stimulus_table.csv`
A table linking image files to beta-response rows.

Required columns:

| column | meaning |
|---|---|
| `stimulus_id` | unique ID for each stimulus |
| `image_path` | path to the image file, relative to `data/` or absolute |
| `category` | optional object/category label |
| `affordance_label` | optional manual affordance label |

#### Brain response files

Each `.npy` file should be a matrix:

```python
n_stimuli × n_voxels
```

Examples:

- `brain_visual.npy`: beta patterns from a visual/object ROI
- `brain_parietal.npy`: beta patterns from a parietal/object-action ROI

The row order must match `stimulus_table.csv`.

In [ ]:
# ============================================================
# 5. Load stimulus table and brain beta responses
# ============================================================

STIMULUS_TABLE_PATH = DATA_DIR / "stimulus_table.csv"
BRAIN_FILES = {
    "Visual_ROI": DATA_DIR / "brain_visual.npy",
    "Parietal_ROI": DATA_DIR / "brain_parietal.npy",
}

# ------------------------------------------------------------------
# Demo mode
# ------------------------------------------------------------------
# Set DEMO_MODE=True if you want to test the notebook without real data.
# This creates fake images and fake brain response matrices.
# For the real project, set DEMO_MODE=False and provide the files above.

DEMO_MODE = True
N_DEMO_STIMULI = 40


def create_demo_data():
    """Create small fake data so the notebook can run end-to-end."""
    img_dir = DATA_DIR / "images"
    img_dir.mkdir(exist_ok=True)

    rows = []
    for i in range(N_DEMO_STIMULI):
        arr = np.random.randint(0, 255, size=(224, 224, 3), dtype=np.uint8)
        img = Image.fromarray(arr)
        fname = f"demo_image_{i:03d}.jpg"
        img.save(img_dir / fname)
        rows.append({
            "stimulus_id": f"stim_{i:03d}",
            "image_path": f"images/{fname}",
            "category": "demo_object",
            "affordance_label": "demo_affordance",
        })

    pd.DataFrame(rows).to_csv(STIMULUS_TABLE_PATH, index=False)

    # Fake beta responses: n_stimuli × n_voxels
    np.save(BRAIN_FILES["Visual_ROI"], np.random.randn(N_DEMO_STIMULI, 300))
    np.save(BRAIN_FILES["Parietal_ROI"], np.random.randn(N_DEMO_STIMULI, 250))


if DEMO_MODE:
    create_demo_data()

stimulus_df = pd.read_csv(STIMULUS_TABLE_PATH)
print(stimulus_df.head())
print("Number of stimuli:", len(stimulus_df))

brain_data = {}
for roi_name, path in BRAIN_FILES.items():
    brain_data[roi_name] = np.load(path)
    print(roi_name, brain_data[roi_name].shape)

# Sanity check: all ROI matrices must have the same number of rows as stimuli.
for roi_name, beta in brain_data.items():
    assert beta.shape[0] == len(stimulus_df), f"Mismatch: {roi_name} has {beta.shape[0]} rows but stimulus table has {len(stimulus_df)} rows"

## 6. Helper functions

The core logic of the project is very compact:

1. Extract ANN features.
2. Compute model RDM.
3. Compute brain RDM.
4. Compare RDMs.

The feature dimension does **not** need to match the voxel dimension. After RDM construction, both become `n_stimuli × n_stimuli` matrices.

In [ ]:
# ============================================================
# 6. Helper functions for images, RDMs, RSA, and plotting
# ============================================================

IMG_TRANSFORM = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])


def resolve_image_path(path_value):
    """Resolve image paths from stimulus_table.csv."""
    p = Path(path_value)
    if p.is_absolute():
        return p
    return DATA_DIR / p


def load_image_batch(image_paths, transform=IMG_TRANSFORM):
    imgs = []
    for p in image_paths:
        img = Image.open(p).convert("RGB")
        imgs.append(transform(img))
    return torch.stack(imgs, dim=0)


def compute_rdm(X, metric="correlation"):
    """Compute an RDM from a matrix of shape n_stimuli × n_features."""
    X = np.asarray(X)
    assert X.ndim == 2, "X must be n_stimuli × n_features"
    distances = pdist(X, metric=metric)
    return squareform(distances)


def upper_triangle_vector(rdm):
    """Flatten upper triangle of a square RDM, excluding diagonal."""
    rdm = np.asarray(rdm)
    assert rdm.shape[0] == rdm.shape[1], "RDM must be square"
    idx = np.triu_indices(rdm.shape[0], k=1)
    return rdm[idx]


def compare_rdms(model_rdm, brain_rdm, method="spearman"):
    """Compare two RDMs using Spearman or Pearson correlation."""
    v1 = upper_triangle_vector(model_rdm)
    v2 = upper_triangle_vector(brain_rdm)

    valid = np.isfinite(v1) & np.isfinite(v2)
    v1 = v1[valid]
    v2 = v2[valid]

    if method == "spearman":
        r, p = spearmanr(v1, v2)
    elif method == "pearson":
        r, p = pearsonr(v1, v2)
    else:
        raise ValueError("method must be 'spearman' or 'pearson'")
    return r, p


def zscore_features(X):
    """Standardize features across stimuli."""
    return StandardScaler().fit_transform(X)


def maybe_pca(X, n_components=100):
    """Apply PCA only if features exceed n_components."""
    X = np.asarray(X)
    max_components = min(X.shape[0] - 1, X.shape[1], n_components)
    if max_components < 2:
        return X
    if X.shape[1] <= max_components:
        return X
    return PCA(n_components=max_components, random_state=SEED).fit_transform(X)


def plot_rdm(rdm, title, save_path=None):
    plt.figure(figsize=(5, 4))
    plt.imshow(rdm, aspect="auto")
    plt.colorbar(label="Dissimilarity")
    plt.title(title)
    plt.xlabel("Stimuli")
    plt.ylabel("Stimuli")
    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()


def plot_rsa_results(results_df, save_path=None):
    plt.figure(figsize=(8, 5))
    for (model, roi), sub in results_df.groupby(["model", "roi"]):
        sub = sub.sort_values("layer_order")
        label = f"{model} — {roi}"
        plt.plot(sub["layer"], sub["rsa_r"], marker="o", label=label)
    plt.axhline(0, linestyle="--", linewidth=1)
    plt.xlabel("Layer depth")
    plt.ylabel("RSA Spearman r")
    plt.title("Layer-wise model-brain RSA")
    plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()

## 7. Brain RDMs

Here we convert each ROI's beta-response matrix into a brain RDM.

Input shape:

```python
n_stimuli × n_voxels
```

Output shape:

```python
n_stimuli × n_stimuli
```

In [ ]:
# ============================================================
# 7. Compute brain RDMs
# ============================================================

brain_rdms = {}

for roi_name, beta in brain_data.items():
    beta_clean = zscore_features(beta)
    brain_rdms[roi_name] = compute_rdm(beta_clean, metric="correlation")
    print(roi_name, brain_rdms[roi_name].shape)

# Visualize one or more brain RDMs
for roi_name, rdm in brain_rdms.items():
    plot_rdm(rdm, f"Brain RDM: {roi_name}", FIGURES_DIR / f"brain_rdm_{roi_name}.png")

## 8. Model feature extraction

We extract features from early, middle, and late layers of three pretrained models:

| Model | Interpretation |
|---|---|
| ResNet-50 | CNN trained with supervised ImageNet labels |
| DINOv2 | Vision Transformer trained with self-supervision |
| OpenCLIP | Vision-language contrastive model |

### Practical note
If runtime is limited, start with **ResNet-50 only**, then add DINOv2, then OpenCLIP.

In [ ]:
# ============================================================
# 8A. ResNet-50 feature extraction
# ============================================================

class ResNetFeatureExtractor(nn.Module):
    """Extract early/middle/late activations from torchvision ResNet-50."""
    def __init__(self):
        super().__init__()
        weights = tv_models.ResNet50_Weights.IMAGENET1K_V2
        model = tv_models.resnet50(weights=weights)
        model.eval()
        self.stem = nn.Sequential(model.conv1, model.bn1, model.relu, model.maxpool)
        self.layer1 = model.layer1
        self.layer2 = model.layer2
        self.layer3 = model.layer3
        self.layer4 = model.layer4
        self.pool = nn.AdaptiveAvgPool2d((1, 1))

    @torch.no_grad()
    def forward(self, x):
        feats = OrderedDict()
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        feats["early"] = self.pool(x).flatten(1)
        x = self.layer3(x)
        feats["middle"] = self.pool(x).flatten(1)
        x = self.layer4(x)
        feats["late"] = self.pool(x).flatten(1)
        return feats


def extract_resnet_features(image_paths, batch_size=16):
    model = ResNetFeatureExtractor().to(DEVICE)
    all_feats = {"early": [], "middle": [], "late": []}

    for start in tqdm(range(0, len(image_paths), batch_size), desc="ResNet50"):
        batch_paths = image_paths[start:start + batch_size]
        x = load_image_batch(batch_paths).to(DEVICE)
        feats = model(x)
        for layer_name, feat in feats.items():
            all_feats[layer_name].append(feat.cpu().numpy())

    return {k: np.concatenate(v, axis=0) for k, v in all_feats.items()}

In [ ]:
# ============================================================
# 8B. DINOv2 feature extraction
# ============================================================
# This uses torch.hub. It needs internet access the first time it runs.
# In Colab, this usually works.

DINO_TRANSFORM = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])


def load_dino_model(model_name="dinov2_vitb14"):
    model = torch.hub.load("facebookresearch/dinov2", model_name)
    model.eval().to(DEVICE)
    return model


def extract_dino_features(image_paths, batch_size=16, model_name="dinov2_vitb14"):
    """
    Extract approximate early/middle/late features from DINOv2 blocks.

    Note:
    DINOv2 internals can change depending on package version.
    If this fails, use the late-layer output only or adapt the block hook names.
    """
    model = load_dino_model(model_name)
    layers_to_hook = {
        "early": 3,
        "middle": 7,
        "late": 11,
    }
    activations = {}
    hooks = []

    def make_hook(name):
        def hook(module, inp, out):
            # ViT block output is usually batch × tokens × dim
            if isinstance(out, tuple):
                out = out[0]
            if out.ndim == 3:
                out = out[:, 0, :]  # CLS token
            activations[name] = out.detach()
        return hook

    for name, idx in layers_to_hook.items():
        hooks.append(model.blocks[idx].register_forward_hook(make_hook(name)))

    all_feats = {"early": [], "middle": [], "late": []}
    for start in tqdm(range(0, len(image_paths), batch_size), desc="DINOv2"):
        batch_paths = image_paths[start:start + batch_size]
        x = load_image_batch(batch_paths, transform=DINO_TRANSFORM).to(DEVICE)
        activations.clear()
        with torch.no_grad():
            _ = model(x)
        for layer_name in all_feats.keys():
            all_feats[layer_name].append(activations[layer_name].cpu().numpy())

    for h in hooks:
        h.remove()

    return {k: np.concatenate(v, axis=0) for k, v in all_feats.items()}

In [ ]:
# ============================================================
# 8C. OpenCLIP feature extraction
# ============================================================
# In Colab, install first if needed:
# !pip install -q open_clip_torch


def extract_openclip_features(image_paths, batch_size=16,
                              model_name="ViT-B-32",
                              pretrained="laion2b_s34b_b79k"):
    """
    Extract early/middle/late features from OpenCLIP ViT.

    If hooks fail due to package version changes, the fallback is to use
    encode_image for the late representation only.
    """
    try:
        import open_clip
    except ImportError as e:
        raise ImportError("Please install open_clip_torch: pip install open_clip_torch") from e

    model, _, preprocess = open_clip.create_model_and_transforms(
        model_name, pretrained=pretrained, device=DEVICE
    )
    model.eval()

    # Use OpenCLIP's own preprocessing pipeline.
    def load_openclip_batch(batch_paths):
        imgs = []
        for p in batch_paths:
            img = Image.open(p).convert("RGB")
            imgs.append(preprocess(img))
        return torch.stack(imgs, dim=0)

    all_feats = {"early": [], "middle": [], "late": []}

    # Try to hook transformer residual blocks.
    activations = {}
    hooks = []
    try:
        blocks = model.visual.transformer.resblocks
        n_blocks = len(blocks)
        layers_to_hook = {
            "early": max(0, n_blocks // 3 - 1),
            "middle": max(0, 2 * n_blocks // 3 - 1),
            "late": n_blocks - 1,
        }

        def make_hook(name):
            def hook(module, inp, out):
                # OpenCLIP ViT may use sequence × batch × dim or batch × sequence × dim.
                if isinstance(out, tuple):
                    out = out[0]
                out = out.detach()
                if out.ndim == 3:
                    # Heuristic: choose CLS-like token dimension.
                    if out.shape[0] < out.shape[1]:
                        # sequence × batch × dim
                        out = out[0]
                    else:
                        # batch × sequence × dim
                        out = out[:, 0, :]
                activations[name] = out
            return hook

        for name, idx in layers_to_hook.items():
            hooks.append(blocks[idx].register_forward_hook(make_hook(name)))

        for start in tqdm(range(0, len(image_paths), batch_size), desc="OpenCLIP"):
            batch_paths = image_paths[start:start + batch_size]
            x = load_openclip_batch(batch_paths).to(DEVICE)
            activations.clear()
            with torch.no_grad():
                _ = model.encode_image(x)
            for layer_name in all_feats.keys():
                all_feats[layer_name].append(activations[layer_name].cpu().numpy())

        for h in hooks:
            h.remove()

        return {k: np.concatenate(v, axis=0) for k, v in all_feats.items()}

    except Exception as exc:
        print("OpenCLIP hook extraction failed:", repr(exc))
        print("Falling back to late encode_image features only.")
        for h in hooks:
            h.remove()

        late_feats = []
        for start in tqdm(range(0, len(image_paths), batch_size), desc="OpenCLIP late only"):
            batch_paths = image_paths[start:start + batch_size]
            x = load_openclip_batch(batch_paths).to(DEVICE)
            with torch.no_grad():
                feat = model.encode_image(x)
            late_feats.append(feat.cpu().numpy())
        late_feats = np.concatenate(late_feats, axis=0)
        return {"early": late_feats, "middle": late_feats, "late": late_feats}

## 9. Run feature extraction

For a fast first pass, run only ResNet-50. Then set `RUN_DINO=True` and `RUN_OPENCLIP=True` once the pipeline works.

In [ ]:
# ============================================================
# 9. Run feature extraction
# ============================================================

image_paths = [resolve_image_path(p) for p in stimulus_df["image_path"].tolist()]

RUN_RESNET = True
RUN_DINO = False       # Set True for real project run
RUN_OPENCLIP = False   # Set True for real project run

model_features = {}

if RUN_RESNET:
    model_features["ResNet50"] = extract_resnet_features(image_paths, batch_size=16)

if RUN_DINO:
    model_features["DINOv2"] = extract_dino_features(image_paths, batch_size=16)

if RUN_OPENCLIP:
    model_features["OpenCLIP"] = extract_openclip_features(image_paths, batch_size=16)

# Print feature shapes
for model_name, layer_dict in model_features.items():
    print("\n", model_name)
    for layer_name, X in layer_dict.items():
        print(f"  {layer_name}: {X.shape}")

## 10. Model RDMs

Here, each model/layer feature matrix is converted into an RDM.

Again, feature dimensions can differ across models:

- ResNet might produce 512/1024/2048-dimensional features.
- DINOv2 might produce 768-dimensional features.
- OpenCLIP might produce 512-dimensional features.

This is fine, because RSA compares **RDMs**, not raw features.

In [ ]:
# ============================================================
# 10. Compute model RDMs
# ============================================================

model_rdms = {}

for model_name, layer_dict in model_features.items():
    model_rdms[model_name] = {}
    for layer_name, X in layer_dict.items():
        X_clean = zscore_features(X)
        X_clean = maybe_pca(X_clean, n_components=100)
        rdm = compute_rdm(X_clean, metric="correlation")
        model_rdms[model_name][layer_name] = rdm
        print(model_name, layer_name, rdm.shape)

# Example RDM heatmaps
for model_name, layer_dict in model_rdms.items():
    for layer_name, rdm in layer_dict.items():
        plot_rdm(rdm, f"Model RDM: {model_name} — {layer_name}",
                 FIGURES_DIR / f"model_rdm_{model_name}_{layer_name}.png")

## 11. RSA: model RDMs vs brain RDMs

For each combination of:

```text
model × layer × ROI
```

we compare the upper triangle of the model RDM with the upper triangle of the brain RDM using Spearman correlation.

In [ ]:
# ============================================================
# 11. Compute RSA
# ============================================================

layer_order_map = {"early": 0, "middle": 1, "late": 2}
rows = []

for model_name, layer_dict in model_rdms.items():
    for layer_name, model_rdm in layer_dict.items():
        for roi_name, brain_rdm in brain_rdms.items():
            r, p = compare_rdms(model_rdm, brain_rdm, method="spearman")
            rows.append({
                "model": model_name,
                "layer": layer_name,
                "layer_order": layer_order_map.get(layer_name, np.nan),
                "roi": roi_name,
                "rsa_r": r,
                "p_value": p,
            })

rsa_results = pd.DataFrame(rows)
rsa_results = rsa_results.sort_values(["roi", "model", "layer_order"])
rsa_results.to_csv(RESULTS_DIR / "rsa_results.csv", index=False)
rsa_results

In [ ]:
# ============================================================
# 12. Main result plot
# ============================================================

plot_rsa_results(rsa_results, save_path=FIGURES_DIR / "main_layerwise_rsa.png")

## 13. Optional: bootstrap confidence intervals

A simple bootstrap over RDM entries gives a rough uncertainty estimate. This is not a perfect substitute for subject-level statistics, but it is useful for a class project and for checking result stability.

If you have multiple subjects, prefer computing RSA per subject and then testing across subjects.

In [ ]:
# ============================================================
# 13. Bootstrap RSA confidence intervals
# ============================================================

def bootstrap_rdm_correlation(model_rdm, brain_rdm, n_boot=1000, method="spearman", seed=SEED):
    rng = np.random.default_rng(seed)
    v1 = upper_triangle_vector(model_rdm)
    v2 = upper_triangle_vector(brain_rdm)
    valid = np.isfinite(v1) & np.isfinite(v2)
    v1, v2 = v1[valid], v2[valid]
    n = len(v1)
    boot = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        if method == "spearman":
            r, _ = spearmanr(v1[idx], v2[idx])
        else:
            r, _ = pearsonr(v1[idx], v2[idx])
        boot.append(r)
    return np.array(boot)

RUN_BOOTSTRAP = False

if RUN_BOOTSTRAP:
    boot_rows = []
    for model_name, layer_dict in model_rdms.items():
        for layer_name, model_rdm in layer_dict.items():
            for roi_name, brain_rdm in brain_rdms.items():
                boot = bootstrap_rdm_correlation(model_rdm, brain_rdm, n_boot=500)
                boot_rows.append({
                    "model": model_name,
                    "layer": layer_name,
                    "roi": roi_name,
                    "mean": np.mean(boot),
                    "ci_low": np.percentile(boot, 2.5),
                    "ci_high": np.percentile(boot, 97.5),
                })
    boot_df = pd.DataFrame(boot_rows)
    boot_df.to_csv(RESULTS_DIR / "rsa_bootstrap_ci.csv", index=False)
    display(boot_df)

## 14. Interpretation template

After running the notebook with real data, fill in this section based on the result table and figure.

### Expected pattern

- Visual ROI: stronger alignment with early/middle visual representations.
- Parietal ROI: stronger alignment with later, more semantic/object-level representations.
- Model comparison: OpenCLIP may show stronger late-layer similarity to higher-level/parietal representations than DINOv2 or ResNet-50.

### Result summary

Replace this text after running the analysis:

```text
In the visual ROI, the strongest RSA was observed for: ...
In the parietal ROI, the strongest RSA was observed for: ...
Across models, the strongest late-layer alignment was observed for: ...
```

### Interpretation

If the late layers of DINOv2 or OpenCLIP show stronger parietal alignment than ResNet-50, this would suggest that architecture and/or training objective shapes the emergence of higher-level object representations.

If OpenCLIP outperforms DINOv2 specifically in the parietal/object-action ROI, this may suggest that vision-language training contributes to representations that go beyond visual shape and category structure.

## 15. Limitations

- This project uses preprocessed beta responses rather than modeling the full raw fMRI pipeline.
- ROI labels may be approximate depending on the dataset. If true LOC/aIPS masks are unavailable, we interpret the ROIs more conservatively as visual-object and parietal ROIs.
- Affordance is operationalized indirectly through ROI alignment. A stronger test would include an explicit affordance model RDM based on manual action labels.
- The three ANNs differ not only in architecture/training objective, but also in pretraining data scale and content.
- Results should be interpreted as representational similarity, not causal evidence that a model “understands” affordances.

## 16. Future directions

- Add an explicit affordance RDM based on labels such as grasp type, action verb, or manipulability.
- Test additional models such as SigLIP, EVA-CLIP, SAM, or embodied/action-trained networks.
- Run the analysis across multiple subjects and perform subject-level statistics.
- Compare RSA with encoding models to test whether the same model layers can also predict voxel responses.
- Extend the analysis to other ROIs such as ventral temporal cortex, premotor cortex, or intraparietal sulcus subdivisions.

## 17. Final take-home message

This notebook tests whether ANN representations become progressively more brain-like in regions associated with visual object processing and object-action understanding.

The key output is a layer-wise RSA comparison:

```text
model × layer × ROI
```

This allows us to ask whether the representational trajectory moves from visual appearance toward more abstract, action-relevant object representations.